MD# SpaceX Falcon 9 – Interactive Visual Analytics with Folium

In [1]:
import folium, pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [2]:
import os
# Loads the local copy (data/) if present, otherwise the IBM Skills Network URL
spacex_df = pd.read_csv('data/spacex_launch_geo.csv') if os.path.exists('data/spacex_launch_geo.csv') else pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site','Lat','Long','class']]
launch_sites_df = spacex_df.groupby('Launch Site', as_index=False).first()[['Launch Site','Lat','Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610746


MD### Task 1 – Mark all launch sites on a map

In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)
for _, r in launch_sites_df.iterrows():
    folium.Circle([r.Lat, r.Long], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    folium.map.Marker([r.Lat, r.Long], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html=f"<div style='font-size:12px;color:#d35400;'><b>{r['Launch Site']}</b></div>")).add_to(site_map)
site_map

MD### Task 2 – Mark the success/failed launches for each site (MarkerCluster)

In [4]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0: 'red'})
site_map.add_child(marker_cluster)
for _, r in spacex_df.iterrows():
    folium.Marker([r.Lat, r.Long], icon=folium.Icon(color='white', icon_color=r.marker_color)).add_to(marker_cluster)
site_map

MD### Task 3 – Distances between a launch site and its proximities

In [5]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2-lat1)/2)**2 + cos(lat1)*cos(lat2)*sin((lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

MousePosition(position='topright', separator=' Long: ', prefix='Lat:', num_digits=20).add_to(site_map)

# CCAFS SLC-40 and nearby points (coordinates read with MousePosition)
launch_site = launch_sites_df[launch_sites_df['Launch Site']=='CCAFS SLC-40'].iloc[0]
proximities = {'Coastline': (28.56367, -80.57163), 'Railway': (28.57206, -80.58526), 'Highway': (28.56335, -80.57085), 'City (Melbourne)': (28.10473, -80.64531)}
for name, (lat, lon) in proximities.items():
    d = calculate_distance(launch_site.Lat, launch_site.Long, lat, lon)
    folium.Marker([lat, lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0),
        html=f"<div style='font-size:12px;color:#d35400;'><b>{name}: {d:.2f} km</b></div>")).add_to(site_map)
    folium.PolyLine([[launch_site.Lat, launch_site.Long], [lat, lon]], weight=1).add_to(site_map)
    print(f'{name}: {d:.2f} km')
site_map.save('launch_sites_map.html')
site_map

Coastline: 0.51 km
Railway: 1.29 km
Highway: 0.58 km
City (Melbourne): 51.43 km
